# 06 — Array cross-correlation and source directions


Reproduce and extend the MATLAB array workflow in Python.

The primary catalogue result is a plane-wave back azimuth and apparent
speed for every event. A fixed-source spherical solution is retained as a
secondary physical check. The two capsule pulses are analyzed separately.


In [1]:

from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from project_config import ensure_output_dirs

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [2]:

from obspy import read
from array_analysis import analyze_array_event

st_corr = read(str(DERIVED_DIR / "bchh_corrected_analysis_window.pkl"),
               format="PICKLE")
geometry = pd.read_csv(DERIVED_DIR / "bchh_geometry.csv")
infra = geometry.loc[geometry["channel"].isin(["HD1", "HD2", "HD3"])]
SENSOR_XY_M = {
    row.channel: (row.easting_m, row.northing_m)
    for row in infra.itertuples(index=False)
}

# SLC-40 location should be replaced with the final surveyed/projected value.
SOURCE_XY_M = None


## Preprocessing for timing

In [3]:

def prepare_timing_stream(stream, starttime, endtime,
                          freqmin=6.0, freqmax=40.0,
                          corners=4, zerophase=True):
    out = stream.select(channel="HD?").copy()
    out.trim(starttime, endtime, pad=False)
    out.detrend("demean")
    out.detrend("linear")
    out.taper(max_percentage=0.05)
    out.filter(
        "bandpass",
        freqmin=freqmin,
        freqmax=freqmax,
        corners=corners,
        zerophase=zerophase,
    )
    return out


## Validate on the key arrivals

In [4]:

key_windows = {
    "Initial second-stage failure": (
        UTCDateTime("2016-09-01T13:07:15.70"),
        UTCDateTime("2016-09-01T13:07:16.30"),
    ),
    "Principal explosion": (
        UTCDateTime("2016-09-01T13:07:18.60"),
        UTCDateTime("2016-09-01T13:07:19.50"),
    ),
    "Capsule pulse 1": (
        UTCDateTime("2016-09-01T13:07:28.30"),
        UTCDateTime("2016-09-01T13:07:28.75"),
    ),
    "Capsule pulse 2": (
        UTCDateTime("2016-09-01T13:07:28.85"),
        UTCDateTime("2016-09-01T13:07:29.25"),
    ),
}

rows = []
pairwise_tables = []

for event_name, (t0, t1) in key_windows.items():
    timing_stream = prepare_timing_stream(st_corr, t0, t1)
    result = analyze_array_event(
        timing_stream,
        SENSOR_XY_M,
        max_lag_s=0.15,
        fixed_source_xy_m=SOURCE_XY_M,
    )
    plane = result["plane_wave"]
    rows.append({
        "event": event_name,
        "back_azimuth_deg": plane["back_azimuth_deg"],
        "apparent_speed_mps": plane["apparent_speed_mps"],
        "rms_lag_residual_s": plane["rms_lag_residual_s"],
        "mean_abs_correlation": (
            result["pairwise_lags"]["abs_correlation"].mean()
        ),
    })
    pairwise = result["pairwise_lags"].copy()
    pairwise.insert(0, "event", event_name)
    pairwise_tables.append(pairwise)

key_array_results = pd.DataFrame(rows)
key_pairwise_lags = pd.concat(pairwise_tables, ignore_index=True)
display(key_array_results)
display(key_pairwise_lags)

key_array_results.to_csv(
    DERIVED_DIR / "key_event_array_results.csv", index=False
)
key_pairwise_lags.to_csv(
    DERIVED_DIR / "key_event_pairwise_lags.csv", index=False
)


,event,back_azimuth_deg,apparent_speed_mps,rms_lag_residual_s,mean_abs_correlation
0,Initial second-stage failure,201.057183,357.827181,0.000021,0.799531
1,Principal explosion,36.139619,387.726458,0.000103,0.734887
2,Capsule pulse 1,201.483552,351.768276,0.000030,0.976393
3,Capsule pulse 2,201.291863,353.500980,0.000066,0.926902


,event,channel_i,channel_j,lag_s,correlation,abs_correlation
0,Initial second-stage failure,HD1,HD2,-0.086190,0.699817,0.699817
1,Initial second-stage failure,HD1,HD3,-0.076345,0.743854,0.743854
2,Initial second-stage failure,HD2,HD3,0.009905,0.954923,0.954923
3,Principal explosion,HD1,HD2,0.060361,0.612809,0.612809
4,Principal explosion,HD1,HD3,0.070872,0.612426,0.612426
5,Principal explosion,HD2,HD3,0.010802,0.979426,0.979426
6,Capsule pulse 1,HD1,HD2,-0.087205,0.972622,0.972622
7,Capsule pulse 1,HD1,HD3,-0.077699,0.971512,0.971512
8,Capsule pulse 1,HD2,HD3,0.009415,0.985044,0.985044
9,Capsule pulse 2,HD1,HD2,-0.087053,0.898878,0.898878


## Extend to the complete event catalogue

In [5]:

EVENT_CATALOGUE_FILE = DERIVED_DIR / "event_catalogue.csv"

if not EVENT_CATALOGUE_FILE.exists():
    print(
        "Place the finalized 153-event catalogue at:\n ",
        EVENT_CATALOGUE_FILE,
        "\nExpected columns should include event_id, window_start, and window_end."
    )
else:
    catalogue = pd.read_csv(EVENT_CATALOGUE_FILE)
    catalogue_rows = []
    for row in catalogue.itertuples(index=False):
        t0, t1 = UTCDateTime(row.window_start), UTCDateTime(row.window_end)
        timing_stream = prepare_timing_stream(st_corr, t0, t1)
        result = analyze_array_event(
            timing_stream,
            SENSOR_XY_M,
            max_lag_s=0.15,
        )
        plane = result["plane_wave"]
        catalogue_rows.append({
            "event_id": row.event_id,
            "window_start": row.window_start,
            "window_end": row.window_end,
            "back_azimuth_deg": plane["back_azimuth_deg"],
            "apparent_speed_mps": plane["apparent_speed_mps"],
            "rms_lag_residual_s": plane["rms_lag_residual_s"],
            "mean_abs_correlation": (
                result["pairwise_lags"]["abs_correlation"].mean()
            ),
        })
    catalogue_results = pd.DataFrame(catalogue_rows)
    catalogue_results.to_csv(
        DERIVED_DIR / "event_catalogue_array_results.csv",
        index=False,
    )
    display(catalogue_results.head())


Place the finalized 153-event catalogue at:
  /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/event_catalogue.csv 
Expected columns should include event_id, window_start, and window_end.
